In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv("/content/spotify_customer_classifiedtwice.csv")

# Remove missing values
df = df.dropna(subset=["customer_tweet", "intent"])

print("Total examples:", len(df))


# ============================================================
# INPUT AND TARGET
# ============================================================

X = df["customer_tweet"].astype(str)
y = df["intent"].astype(str)


# ============================================================
# TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training examples:", len(X_train))
print("Testing examples:", len(X_test))


# ============================================================
# TF-IDF
# ============================================================

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("\nTF-IDF vocabulary size:", len(vectorizer.vocabulary_))


# ============================================================
# LOGISTIC REGRESSION
# ============================================================

model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train_tfidf, y_train)


# ============================================================
# PREDICTIONS
# ============================================================

y_pred = model.predict(X_test_tfidf)


# ============================================================
# EVALUATION
# ============================================================

accuracy = accuracy_score(y_test, y_pred)

print("\n========================================")
print("BASELINE 2: TF-IDF + LOGISTIC REGRESSION")
print("========================================")

print("\nAccuracy:", round(accuracy, 4))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)


# ============================================================
# CONFUSION MATRIX
# ============================================================

labels = sorted(y.unique())

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=labels
)

print("\nConfusion Matrix:")

cm_df = pd.DataFrame(
    cm,
    index=labels,
    columns=labels
)

display(cm_df)


# ============================================================
# TEST WITH A NEW CUSTOMER TWEET
# ============================================================

test_tweet = "my spotify keeps stopping when I play songs"

test_vector = vectorizer.transform([test_tweet])

predicted_intent = model.predict(test_vector)[0]

# Probability / confidence
probabilities = model.predict_proba(test_vector)[0]

confidence = probabilities.max()

print("\n========================================")
print("NEW CUSTOMER TEST")
print("========================================")

print("\nCustomer tweet:")
print(test_tweet)

print("\nPredicted intent:")
print(predicted_intent)

print("\nConfidence:")
print(round(confidence, 4))

Total examples: 14639
Training examples: 11711
Testing examples: 2928

TF-IDF vocabulary size: 18599

BASELINE 2: TF-IDF + LOGISTIC REGRESSION

Accuracy: 0.8108

Classification Report:
                             precision    recall  f1-score   support

        app_technical_issue       0.81      0.83      0.82       633
        audio_quality_issue       1.00      0.04      0.08        51
            billing_payment       0.95      0.45      0.61        86
       cancellation_request       1.00      0.07      0.13        14
                  complaint       0.00      0.00      0.00        20
           feature_question       0.88      0.39      0.54       127
        login_account_issue       0.87      0.70      0.77       165
missing_unavailable_content       0.50      0.04      0.07        27
                      other       0.79      0.99      0.88      1375
             playback_issue       0.87      0.79      0.83       335
             playlist_issue       1.00      0.13      0

,app_technical_issue,audio_quality_issue,billing_payment,cancellation_request,complaint,feature_question,login_account_issue,missing_unavailable_content,other,playback_issue,playlist_issue,premium_subscription
app_technical_issue,523,0,0,0,0,1,1,0,101,7,0,0
audio_quality_issue,18,2,0,0,0,1,0,0,27,3,0,0
billing_payment,4,0,39,0,0,0,9,1,29,4,0,0
cancellation_request,1,0,0,1,0,0,6,0,5,0,0,1
complaint,0,0,0,0,0,0,0,0,19,1,0,0
feature_question,4,0,0,0,0,50,0,0,69,4,0,0
login_account_issue,19,0,2,0,0,1,115,0,24,4,0,0
missing_unavailable_content,8,0,0,0,0,1,0,1,14,2,0,1
other,8,0,0,0,0,0,1,0,1361,5,0,0
playback_issue,31,0,0,0,0,0,0,0,40,264,0,0



NEW CUSTOMER TEST

Customer tweet:
my spotify keeps stopping when I play songs

Predicted intent:
playback_issue

Confidence:
0.9293
